In [ ]:
# Read raw sales from ADLS Bronze
storage_account = 'adlsretailpulse<initials>'
container       = 'retaildata'

df_raw = spark.read.option('header', True) \
              .option('inferSchema', True) \
              .csv(f'abfss://{container}@{storage_account}.dfs.core.windows.net/raw/sales/')

print(f'Row count: {df_raw.count():,}')   # Expected: ~500,000
df_raw.printSchema()

StatementMeta(sparkRetail, 9, 2, Finished, Available, Finished, False)

Row count: 500,000
root
 |-- transaction_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- region: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- total_amount: double (nullable = true)



In [ ]:
# Data Quality: inspect nulls
from pyspark.sql.functions import col, sum as _sum

null_counts = df_raw.select([_sum(col(c).isNull().cast('int')).alias(c)
                             for c in df_raw.columns])
null_counts.show()
# Expected: discount_pct should show ~15,000 nulls (3% of 500k)

StatementMeta(sparkRetail, 9, 3, Finished, Available, Finished, False)

+--------------+----------------+--------+-----------+----------+--------+----------+------------+------+--------------+------------+
|transaction_id|transaction_date|store_id|customer_id|product_id|quantity|unit_price|discount_pct|region|payment_method|total_amount|
+--------------+----------------+--------+-----------+----------+--------+----------+------------+------+--------------+------------+
|             0|               0|       0|          0|         0|       0|         0|       15000|     0|             0|           0|
+--------------+----------------+--------+-----------+----------+--------+----------+------------+------+--------------+------------+



In [ ]:
# Cleaning & Type Casting
from pyspark.sql.functions import to_date, when, round as _round

df_clean = (df_raw
    .withColumn('transaction_date', to_date('transaction_date', 'yyyy-MM-dd'))
    .withColumn('discount_pct', when(col('discount_pct').isNull(), 0)
                                .otherwise(col('discount_pct').cast('double')))
    .withColumn('unit_price',   col('unit_price').cast('double'))
    .withColumn('quantity',     col('quantity').cast('int'))
    .withColumn('total_amount', _round(col('quantity') * col('unit_price')
                                * (1 - col('discount_pct')/100), 2))
    .dropDuplicates(['transaction_id'])
    .filter(col('unit_price') > 0)
)

print(f'Clean rows: {df_clean.count():,}')

StatementMeta(sparkRetail, 9, 4, Finished, Available, Finished, False)

Clean rows: 500,000


# Cell 5 — Write Silver Parquet (partitioned)
silver_path = f'abfss://{container}@{storage_account}.dfs.core.windows.net/curated/sales/'

df_enriched.write \
    .mode('overwrite') \
    .partitionBy('year', 'month') \
    .parquet(silver_path)

print('Silver layer written successfully!')
# Expected: curated/sales/year=2023/month=1/ ... month=12/ folders in ADLS